<a href="https://colab.research.google.com/github/leweieeeee/Transformer-Warm-Start-ADAPT-VQE/blob/main/H4_ADAPT_VQE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

In [ ]:
%pip install pennylane
%pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 49.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user
^C


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pennylane as qml

from pennylane import RandomLayers
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import numpy as np
import time
import pandas as pd

from pennylane import qchem

ModuleNotFoundError: No module named 'pennylane'

In [ ]:
def generate_data(bond_lengths, active_electrons=4, active_orbitals=4, basis="sto-3g"):
    data = []

    for L in bond_lengths:
        symbols = ["H", "H", "H", "H"]
        geometry = np.array([
            [0.0, 0.0, 0.0],
            [0.0, 0.0, float(L)],
            [0.0, 0.0, 2.0 * float(L)],
            [0.0, 0.0, 3.0 * float(L)],
        ], dtype=float)
        molecule = qchem.Molecule(symbols, geometry)

        H, qubits = qchem.molecular_hamiltonian(
            molecule,
            active_electrons=active_electrons,
            active_orbitals=active_orbitals,
        )

        singles, doubles = qchem.excitations(active_electrons, qubits)

        data.append({
            "bond_length": float(L),
            "molecule": molecule,
            "hamiltonian": H,
            "qubits": qubits,
            "singles": singles,
            "doubles": doubles,
        })

    return data


In [ ]:
def adapt_vqe(H, qubits, singles, doubles, max_iter=20):
    active_electrons = 4
    hf_state = qchem.hf_state(active_electrons, qubits)
    dev = qml.device("default.qubit", wires=qubits)

    @qml.qnode(dev)
    def circuit():
        [qml.PauliX(i) for i in np.nonzero(hf_state)[0]]
        return qml.expval(H)

    singles_excitations = [qml.SingleExcitation(0.0, x) for x in singles]
    doubles_excitations = [qml.DoubleExcitation(0.0, x) for x in doubles]
    operator_pool = doubles_excitations + singles_excitations
    opt = qml.optimize.AdaptiveOptimizer()
    records = []

    for step in range(min(max_iter, len(operator_pool))):
        start_time = time.time()

        circuit, energy, gradient = opt.step_and_cost(circuit, operator_pool)

        elapsed_time = time.time() - start_time

        print(f"n = {step},  E = {energy:.8f} H, Largest Gradient = {gradient:.3f}")
        print(qml.draw(circuit, decimals=None)(), "\n")

        tape = circuit.construct([], {})
        for op in tape.operations:
            print(op.name, op.wires, op.parameters)

        for idx, op in enumerate(tape.operations):
            record = {
                "step": step,
                "gate_type": op.name,
                "depth": idx,
                "energy": energy,
                "time": elapsed_time,
                "param": float(op.parameters[0]) if op.parameters else None
            }
            records.append(record)

        if gradient < 3e-4:
            break

    df = pd.DataFrame(records)
    return df

In [ ]:
bond_lengths = jnp.linspace(0.7, 3.0, 5)
dataset = generate_data(bond_lengths)
dfs = []

for entry in dataset:
    L        = entry["bond_length"]
    H        = entry["hamiltonian"]
    qubits   = entry["qubits"]
    singles  = entry["singles"]
    doubles  = entry["doubles"]
    df = adapt_vqe(H, qubits, singles, doubles)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

combined_df.to_csv("H4_combined_dataset.csv",index = False)


In [ ]:
df_store = df.drop(columns = ["energy","time"])
df_store.to_csv("H4_store_dataset.csv", index=False)